<a href="https://colab.research.google.com/github/Shubham-Chavan-7/Sales-Analysis/blob/main/House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

In [4]:
DATA_PATH = "/content/HousePrediction.xlsx"
OUTPUT_DIR = "./house_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42

In [5]:
print("Loading dataset from:", DATA_PATH)
df = pd.read_excel(DATA_PATH)


Loading dataset from: /content/HousePrediction.xlsx


In [6]:
# Display all columns and first 100 rows
pd.set_option('display.max_columns', None)
print("\nAll columns:")
print(df.columns.tolist())

print("\nFirst 10 rows (preview):")
print(df.head(10))

print("\nFirst 100 rows (if dataset >= 100 rows):")
print(df.head(100))

# Column insights: data types and basic non-null counts
print("\nDataframe info:")
print(df.info())

print("\nSummary (describe) of numerical columns:")
print(df.describe(include=[np.number]).T)



All columns:
['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'LotConfig', 'BldgType', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'Exterior1st', 'BsmtFinSF2', 'TotalBsmtSF', 'SalePrice']

First 10 rows (preview):
   Id  MSSubClass MSZoning  LotArea LotConfig BldgType  OverallCond  \
0   0          60       RL     8450    Inside     1Fam            5   
1   1          20       RL     9600       FR2     1Fam            8   
2   2          60       RL    11250    Inside     1Fam            5   
3   3          70       RL     9550    Corner     1Fam            5   
4   4          60       RL    14260       FR2     1Fam            5   
5   5          50       RL    14115    Inside     1Fam            5   
6   6          20       RL    10084    Inside     1Fam            5   
7   7          60       RL    10382    Corner     1Fam            6   
8   8          50       RM     6120    Inside     1Fam            5   
9   9         190       RL     7420    Corner   2fmCon            6   

   YearB

In [7]:
# === 2) Task 2: Missing values and analysis ===
print("\n=== Missing values per column ===")
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing)

# Columns with NaNs
columns_with_nan = missing.index.tolist()
print("\nColumns with missing values:", columns_with_nan)

# For each column with missing values: compute mean SalePrice when present vs missing
if 'SalePrice' not in df.columns:
    raise ValueError("SalePrice column not found in dataset. Please verify the spreadsheet.")

print("\nMean SalePrice when a feature is present vs missing:")
for col in columns_with_nan:
    present_mask = df[col].notna()
    missing_mask = df[col].isna()
    mean_present = df.loc[present_mask, 'SalePrice'].mean()
    mean_missing = df.loc[missing_mask, 'SalePrice'].mean()
    print(f"{col}: mean when present = {mean_present:.2f}, mean when missing = {mean_missing if pd.notna(mean_missing) else 'NA'}")

# Count numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric feature count: {len(numeric_cols)}")
print("Numeric features (first 20):", numeric_cols[:20])

# Print the first five rows of numerical values
print("\nFirst five rows of numeric features:")
print(df[numeric_cols].head())



=== Missing values per column ===
SalePrice      1459
MSZoning          4
BsmtFinSF2        1
TotalBsmtSF       1
Exterior1st       1
dtype: int64

Columns with missing values: ['SalePrice', 'MSZoning', 'BsmtFinSF2', 'TotalBsmtSF', 'Exterior1st']

Mean SalePrice when a feature is present vs missing:
SalePrice: mean when present = 180921.20, mean when missing = NA
MSZoning: mean when present = 180921.20, mean when missing = NA
BsmtFinSF2: mean when present = 180921.20, mean when missing = NA
TotalBsmtSF: mean when present = 180921.20, mean when missing = NA
Exterior1st: mean when present = 180921.20, mean when missing = NA

Numeric feature count: 9
Numeric features (first 20): ['Id', 'MSSubClass', 'LotArea', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF2', 'TotalBsmtSF', 'SalePrice']

First five rows of numeric features:
   Id  MSSubClass  LotArea  OverallCond  YearBuilt  YearRemodAdd  BsmtFinSF2  \
0   0          60     8450            5       2003          2003         0.0  

In [8]:
# === 3) Year features comparison ===
# Create simple age features if YearBuilt and YearRemodAdd present
if 'YearBuilt' in df.columns:
    df['House_Age'] = df['YearBuilt'].max() - df['YearBuilt']  # relative age
if 'YearRemodAdd' in df.columns:
    df['Since_Remod'] = df['YearRemodAdd'].max() - df['YearRemodAdd']

# Scatter plots saved to files
def save_scatter(xcol, ycol='SalePrice', fname=None):
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=df, x=xcol, y=ycol)
    plt.title(f"{ycol} vs {xcol}")
    plt.tight_layout()
    if fname:
        plt.savefig(os.path.join(OUTPUT_DIR, fname))
    plt.close()

year_cols = [c for c in ['YearBuilt', 'YearRemodAdd', 'House_Age', 'Since_Remod'] if c in df.columns]
for c in year_cols:
    save_scatter(c, 'SalePrice', fname=f"scatter_{c}_vs_SalePrice.png")


In [9]:
# === 4) Discrete and continuous relationships ===
# Example: OverallCond (discrete) vs SalePrice
if 'OverallCond' in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='OverallCond', y='SalePrice', data=df)
    plt.title("SalePrice by OverallCond")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "box_OverallCond_vs_SalePrice.png"))
    plt.close()

# Continuous variables correlation with SalePrice
cont_cols = df.select_dtypes(include=[np.number]).columns.drop('Id', errors='ignore').tolist()
cont_cols = [c for c in cont_cols if c != 'SalePrice']
corrs = df[cont_cols + ['SalePrice']].corr()['SalePrice'].sort_values(ascending=False)
print("\nTop correlations with SalePrice:")
print(corrs.head(15))

# Draw heatmap of top numeric correlations
top_corrs = corrs.abs().sort_values(ascending=False).head(15).index.tolist()
plt.figure(figsize=(10,8))
sns.heatmap(df[top_corrs + ['SalePrice']].corr(), annot=True, fmt=".2f", cmap='vlag')
plt.title("Correlation matrix (top features)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "correlation_top_features.png"))
plt.close()

# Histograms for continuous variables
for c in cont_cols:
    plt.figure(figsize=(5,3))
    sns.histplot(df[c].dropna(), kde=False)
    plt.title(f"Histogram of {c}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"hist_{c}.png"))
    plt.close()

# Log transform SalePrice distribution
plt.figure(figsize=(6,4))
sns.histplot(np.log1p(df['SalePrice']), kde=True)
plt.title("Log1p(SalePrice) distribution")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hist_log_SalePrice.png"))
plt.close()

# Add log price column
df['LogSalePrice'] = np.log1p(df['SalePrice'])


Top correlations with SalePrice:
SalePrice       1.000000
TotalBsmtSF     0.613581
YearBuilt       0.522897
YearRemodAdd    0.507101
LotArea         0.263843
BsmtFinSF2     -0.011378
OverallCond    -0.077856
MSSubClass     -0.084284
Since_Remod    -0.507101
House_Age      -0.522897
Name: SalePrice, dtype: float64


In [10]:
# === 5) Outliers detection (simple) ===
# We'll use IQR method on SalePrice and a couple continuous features
def find_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

outliers = find_outliers_iqr(df['SalePrice'])
print(f"\nDetected {len(outliers)} SalePrice outliers by IQR method.")
# Save a boxplot for SalePrice
plt.figure(figsize=(6,3))
sns.boxplot(x=df['SalePrice'])
plt.title("SalePrice Boxplot")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "box_SalePrice.png"))
plt.close()



Detected 61 SalePrice outliers by IQR method.


In [12]:
# === 6) Categorical features and SalePrice relationship ===
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nCategorical columns count: {len(cat_cols)}")
print("Categorical columns (sample):", cat_cols[:20])

# Example groupby mean SalePrice for top categorical cols
for c in cat_cols[:6]:
    print(f"\nTop groups in {c} by mean SalePrice:")
    print(df.groupby(c)['SalePrice'].mean().sort_values(ascending=False).head(10))

# Plot MSZoning vs SalePrice if exists
if 'MSZoning' in df.columns:
    plt.figure(figsize=(8,4))
    order = df.groupby('MSZoning')['SalePrice'].median().sort_values().index
    sns.boxplot(x='MSZoning', y='SalePrice', data=df, order=order)
    plt.title("SalePrice by MSZoning")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "box_MSZoning_vs_SalePrice.png"))
    plt.close()



Categorical columns count: 4
Categorical columns (sample): ['MSZoning', 'LotConfig', 'BldgType', 'Exterior1st']

Top groups in MSZoning by mean SalePrice:
MSZoning
FV         214014.061538
RL         191004.994787
RH         131558.375000
RM         126316.830275
C (all)     74528.000000
Name: SalePrice, dtype: float64

Top groups in LotConfig by mean SalePrice:
LotConfig
CulDSac    223854.617021
FR3        208475.000000
Corner     181623.425856
FR2        177934.574468
Inside     176938.047529
Name: SalePrice, dtype: float64

Top groups in BldgType by mean SalePrice:
BldgType
1Fam      185763.807377
TwnhsE    181959.342105
Twnhs     135911.627907
Duplex    133541.076923
2fmCon    128432.258065
Name: SalePrice, dtype: float64

Top groups in Exterior1st by mean SalePrice:
Exterior1st
ImStucc    262000.000000
Stone      258500.000000
CemntBd    231690.655738
VinylSd    213732.900971
BrkFace    194573.000000
Plywood    175942.379630
HdBoard    163077.450450
Stucco     162990.000000
WdShi